In [1]:
import torch
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
matplotlib.use('Agg')

from dataclasses import dataclass
from torch.distributions import Gamma, Multinomial, Dirichlet, Beta, Binomial
from tqdm.auto import tqdm

import os
import sys
from path import Path
current_directory = os.getcwd()
parent_directory = Path(current_directory).parent
sys.path.append(parent_directory)
from apf.base.sample import Sampler
from GS_DPGM import HGPDR,config,CRT
%load_ext Cython
%load_ext autoreload
%autoreload 2

In [2]:
test = np.load('/home/HuangRui/PointProcess/NodeGroup/data/icews_tensor_preprocessed_33610010013.npz', allow_pickle=True)

In [ ]:
from FS_PRGDS_Interval_Tensor import FS_PRGDS_tensor
# run model
burnin = 600
maxiter = 400
params = {
'tau': 1.,
'alpha0': 10.,
'epsilon_the': 0.,
'epsilon_lam': 1.,
'stationary': True,
'data' : torch.tensor(test['data'][:108]),
'K' : 50, # latent components
'S' : 54, # sub-intervals
'parallel':True
}

fs_prgds = FS_PRGDS_tensor(**params)
expectation_collection = []
transition_steady = []
for iter in tqdm(range(burnin+maxiter)):
    fs_prgds.sample_n()
    fs_prgds.sample_h()
    fs_prgds.sample_the()
    fs_prgds.sample_lam()
    fs_prgds.sample_delt()
    fs_prgds.sample_g()
    fs_prgds.sample_gam()
    fs_prgds.sample_beta()
    fs_prgds.sample_phi()
    fs_prgds.sample_pi()
    fs_prgds.sample_eta()
    fs_prgds.sample_A()
    fs_prgds.sample_dpgm()

    # compute expectation of training data
    part1 = torch.einsum('ik, jk, lk -> ijlk', fs_prgds.phi[0], fs_prgds.phi[1], fs_prgds.phi[2])
    part2 = torch.einsum('k, tk, t -> kt', fs_prgds.lam, fs_prgds.the, fs_prgds.delt)
    expectation = torch.einsum('ijlk, kt -> ijlt', part1, part2)
    expectation_collection.append(expectation.clone())

    if iter > burnin:
        transition_steady.append(fs_prgds.pi.clone())